# Spatial analysis and islet inflammation

Inspect a whole pancreatic section, measure cell neighborhoods from native centroids, and compare islet pseudotime with core inflammation across donors.

The notebooks require the November 13 Islet Explorer protein release that excludes EpCAM from typing and rebuilds the pancreas UMAP without that marker. The load step checks the release ID and the marker list saved with the embedding.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/03_spatial_analysis.ipynb).

## 1. Select a donor

The native-cell graph connects cells within a section. The islet graph has a separate fit for each donor.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "typing_release.json").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the no-EpCAM Scanpy release folder containing typing_release.json.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})

# Locate the shared plotting module when running from a notebook, its parent, or the Senior checkout.
import sys
helper_paths = [cwd, cwd/"notebooks", cwd/"phenocycler"/"notebooks", cwd/"analysis"/"phenocycler"/"notebooks",
                data.parent/"notebooks"]
helper_dir = next((p for p in helper_paths if (p/"phenotype_signatures.py").is_file()), None)
if helper_dir is None:
    raise FileNotFoundError("Run from the Senior checkout or its analysis/phenocycler/notebooks directory.")
sys.path.insert(0, str(helper_dir))
from phenotype_signatures import (PAN_STRUCTURE, HORMONES, IMMUNE, T_CELL, DISPLAY_NAMES,
                                  donor_heatmap, show_profiles, conflict_summary, validate_release)
from phenotype_signatures import (cell_subtypes, phenotype_colors, phenotype_key, names_in_order, subtype_display_name)
release = validate_release(data)

DONOR = "115"                      # Any donor listed in the manifest
DISEASE = "ND"                     # ND, AAb+, or T1D
group_order = ["ND","AAb+","T1D"]
group_ink = {"ND":"#23699b","AAb+":"#ab6c06","T1D":"#ae4643"}
counts = pd.read_parquet(data / "donor_counts.parquet").query("tissue == 'Pancreas'").copy()
evidence = pd.read_parquet(data / "donor_markers.parquet").query("tissue == 'Pancreas'").copy()
counts["donor"] = counts.donor.astype(str)
evidence["donor"] = evidence.donor.astype(str)
manifest = pd.read_csv(data / "pancreas_manifest.csv",dtype={"donor":str})
if DONOR not in set(manifest.donor) or DISEASE not in group_order:
    raise ValueError("Choose a donor and group from pancreas_manifest.csv")
inks = phenotype_colors(counts, "Pancreas")

## 2. Spatial visualization

The most specific recorded cell assignments and islet compartments are shown at their native centroids using a uniform display sample. The CD99 endocrine group with no measurable hormone above the typing floor and the competing-hormone group retain separate labels.

In [ ]:
# Native coordinates, µm. Draw a uniform subset of centroids to keep whole sections legible.
donor_path = data / "pancreas" / f"{DONOR}.h5ad"
donor = sc.read_h5ad(donor_path,backed="r")
assert donor.n_obs == manifest.set_index("donor").at[DONOR,"cells"]
rng = np.random.default_rng(19)
ix = np.sort(rng.choice(donor.n_obs,min(donor.n_obs,60000),replace=False))
labels = cell_subtypes(donor.obs, "Pancreas")
obs = donor.obs.iloc[ix][["islet_region"]].copy()
obs["subtype"] = labels.iloc[ix].cat.remove_unused_categories()
obs["islet_region"] = obs.islet_region.cat.remove_unused_categories()
xy = np.asarray(donor.obsm["spatial"])[ix]
section_sample = ad.AnnData(obs=obs, obsm={"spatial":xy})
zones = {"outside":"#c8c6c1","core":"#126891","peri":"#d78939","multiple peri-islet bands":"#6e4c88"}
shown = names_in_order(obs.subtype.cat.categories, "Pancreas")
legend_height = .24 * np.ceil(len(shown) / 3) + .4
fig = plt.figure(figsize=(13, 5.8 + legend_height), layout="constrained")
grid = fig.add_gridspec(2, 2, height_ratios=[5,legend_height])
for j,(key,palette,title) in enumerate([("subtype",inks,"Cell subtype"),("islet_region",zones,"Islet compartment")]):
    ax = fig.add_subplot(grid[0,j])
    sc.pl.embedding(section_sample,basis="spatial",color=key,ax=ax,show=False,
                    size=3,palette=palette,legend_loc="none",title=title)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    if key == "islet_region":
        handles = [Line2D([], [], marker="o", ms=4, ls="", color=zones[v], label=v) for v in obs.islet_region.cat.categories]
        ax.legend(handles=handles,loc="upper center",bbox_to_anchor=(.5,0),ncol=2,frameon=False,fontsize=8)
phenotype_key(fig.add_subplot(grid[1,:]), shown, inks, columns=3)
fig.suptitle(f"Donor {DONOR} · {donor.n_obs:,} admitted cells · {len(ix):,} uniformly sampled centroids")
plt.show()

## 3. Spatial neighborhood enrichment

Each row is a focal phenotype and each column is a neighbor phenotype. The graph uses every admitted centroid in the selected section. The heatmap shows descriptive enrichment relative to the section abundance. Gray cells have no observed edges or an undefined expectation.

In [ ]:
from scipy.spatial import cKDTree

# Ten nearest other cells, computed within this section from all admitted native centroids.
xy = np.asarray(donor.obsm["spatial"], dtype=np.float64)
labels = cell_subtypes(donor.obs, "Pancreas").cat.remove_unused_categories()
groups = labels.cat.categories
codes = labels.cat.codes.to_numpy(dtype=np.int64)
tree = cKDTree(xy)
k = min(10, len(xy)-1)
pairs = np.zeros((len(groups),len(groups)),dtype=np.int64)
for start in range(0,len(xy),50000):
    stop = min(start+50000,len(xy))
    _, neighbors = tree.query(xy[start:stop],k=k+1, workers=2)
    centers = np.arange(start,stop)[:,None]
    rank = np.cumsum(neighbors!=centers,axis=1)
    take = (neighbors!=centers) & (rank<=k)
    edges = codes[start:stop,None]*len(groups) + codes[neighbors]
    pairs += np.bincount(edges[take],minlength=len(groups)**2).reshape(pairs.shape)
assert pairs.sum() == len(xy)*k
# Enrichment relative to random mixing at the observed section-wide abundance.
# A cell cannot be its own neighbor; use the finite-population expectation.
abundance = np.bincount(codes,minlength=len(groups))
null = (abundance[None,:] - np.eye(len(groups))) / (len(xy)-1)
pobs = pairs / pairs.sum(axis=1,keepdims=True)
ratio = np.log2(np.divide(pobs,null,out=np.full_like(pobs,np.nan),where=(pobs>0)&(null>0)))
order = [groups.get_loc(t) for t in names_in_order(groups, "Pancreas")]
mat = ratio[np.ix_(order,order)]; names=[subtype_display_name(label) for label in groups[order]]
side = max(6.4, .25 * len(names))
fig, ax = plt.subplots(figsize=(side + 4.3, side + 2.2), layout="constrained")
lim=max(1,float(np.nanpercentile(np.abs(mat),95)))
cmap=plt.get_cmap("RdBu_r").copy();cmap.set_bad("#dadada")
im=ax.imshow(mat,cmap=cmap,norm=TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim),aspect="auto")
ax.set(xticks=range(len(names)),yticks=range(len(names)),xticklabels=names,yticklabels=names,
       title=f"Cell neighborhoods · donor {DONOR}",xlabel="Neighbor identity",ylabel="Focal cell identity")
ax.tick_params(axis="both", labelsize=8)
plt.setp(ax.get_xticklabels(),rotation=65,ha="right",rotation_mode="anchor")
fig.colorbar(im,ax=ax,label=r"$\log_2$(observed / expected)",shrink=.72)
plt.show()

## 4. Islet inflammation trajectory within a donor

For each native islet, add the PhenoCycler immune-cell counts inside the islet and in the surrounding 20 µm of pancreatic tissue. Divide by the corresponding measured area in mm². The surrounding band excludes endocrine objects and tissue outside the pancreatic annotation. The table retains the two areas and counts so a small inflamed rim can be distinguished from infiltration of the core.

The inflammation trajectory orders islets by this combined density within each donor. Measured zeros tie at the start and equal densities have equal ranks. The rank describes cross-sectional burden. The accompanying composition DPT uses a separate graph of cell populations and can be compared with the density ordering below.

In [ ]:
islets = sc.read_h5ad(data / "pancreas_islets.h5ad")
density = "combined_immune_per_mm2"
rank = "pseudotime_rank"
required = {density, rank, "core_area_um2", "peri_area_um2", "composition_dpt_rank", "density_eligible"}
missing = required.difference(islets.obs)
if missing:
    raise ValueError(f"Use the density-trajectory release. Missing islet fields: {sorted(missing)}")
section = islets[islets.obs.donor.astype(str).eq(DONOR)].copy()
ordered = section.obs.dropna(subset=[rank, density]).sort_values(density)
print(f"Donor {DONOR} · {len(ordered):,} ordered / {section.n_obs:,} native islets")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for ax, field, title, ink in [
        (axes[0], "core_immune_per_mm2", "Inside the islet", "#246b85"),
        (axes[1], "peri_immune_per_mm2", "In the surrounding 20 µm", "#a15346")]:
    observed = ordered.dropna(subset=[field])
    ax.scatter(observed[rank], observed[field], s=12, alpha=.48, ls="none", c=ink)
    ax.set(xlim=(-.03, 1.03), xlabel="Within-donor inflammation rank", title=title,
           ylabel="Immune cells per mm²", yscale="symlog")
    ax.grid(axis="y", alpha=.2)
fig.suptitle(f"Donor {DONOR} · core and peri-islet inflammation")
fig.tight_layout()
plt.show()
display(section.obs[["islet_id", "n_core_immune", "core_area_um2", "n_peri_immune",
                     "peri_area_um2", density, rank, "density_eligible"]].sort_values(rank).round(3))


## 5. Inflammation trajectories for all donors

Each point represents a native islet with an assessable core and surrounding tissue area. The horizontal axis is the within-donor rank, and the vertical axis gives the combined immune-cell density used to order the islets. Ranks increase with density by construction. Each panel gives its ordered / native islet count. All panels share a density axis in cells/mm².

In [ ]:
from matplotlib.ticker import FuncFormatter

sections = manifest.assign(disease_order=manifest.disease.map({g:i for i,g in enumerate(group_order)}),
                           donor_number=manifest.donor.astype(int)).sort_values(["disease_order","donor_number"])
rows = int(np.ceil(len(sections) / 5))
fig, axes = plt.subplots(rows, 5, figsize=(13.6, 2.1*rows+1.0), sharex=True, sharey=True, squeeze=False)
observed = islets.obs.dropna(subset=[rank, density])
upper = max(10, observed[density].max()) * 1.1
ticks = [v for v in (0,10,100,1000,10000,100000) if v <= upper]
for ax, row in zip(axes.flat, sections.itertuples()):
    all_islets = islets.obs[islets.obs.donor.astype(str).eq(row.donor)]
    z = all_islets.dropna(subset=[rank, density]).sort_values(density)
    ax.set_yscale("symlog", linthresh=10, linscale=.75)
    ax.set(xlim=(-.03,1.03), ylim=(-1,upper), xticks=[0,.5,1], yticks=ticks)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y,pos:f"{y:,.0f}"))
    ax.tick_params(labelsize=7)
    ax.grid(axis="y", color="#e4e7e8", lw=.45)
    if len(z):
        if not z[rank].is_monotonic_increasing or not z.loc[z[density].eq(0), rank].eq(0).all():
            raise ValueError(f"Density and inflammation rank disagree in donor {row.donor}")
        ax.scatter(z[rank], z[density], s=7, lw=0, alpha=.55, color=group_ink[row.disease], rasterized=True)
    else:
        ax.text(.5,.5,"No assessable islets", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(f"{row.donor} · {row.disease} · {len(z):,}/{len(all_islets):,}", fontsize=9, loc="left",
                 color=group_ink[row.disease])
for ax in list(axes.flat)[len(sections):]:
    ax.set_visible(False)
fig.supxlabel("Within-donor inflammation rank", fontsize=11, y=.033)
fig.supylabel("Core + surrounding immune cells per mm² of pancreatic tissue", fontsize=11, x=.012)
fig.suptitle("Islet inflammation trajectories by donor", fontsize=15, y=.99)
fig.subplots_adjust(left=.075, right=.99, top=.94, bottom=.085, wspace=.17, hspace=.37)
plt.show()

## 6. How does the original composition graph relate to inflammation?

The original within-donor diffusion pseudotime (DPT) is rooted using nonimmune composition and reflects several cell populations. Compare it against measured immune-cell density for the islets with both estimates. Each point summarizes one donor. The table also reports islets at zero measured burden and the number eligible for each ordering.

In [ ]:
res = []
for (group, did), frame in islets.obs.groupby(["disease","donor"],observed=True):
    z = frame.dropna(subset=["composition_dpt_rank", density])
    rho = (z.composition_dpt_rank.corr(z[density], method="spearman")
           if z.composition_dpt_rank.nunique()>1 and z[density].nunique()>1 else np.nan)
    res.append({"disease":group, "donor":did, "native_islets":len(frame),
                "inflammation_ordered":frame[rank].count(), "zero_density":frame[density].eq(0).sum(),
                "composition_DPT_measured":len(z), "rho":rho})
overview = pd.DataFrame(res).set_index("donor")
fig, ax = plt.subplots(figsize=(7,3.8))
for i,g in enumerate(group_order):
    y=overview.loc[overview.disease.eq(g),"rho"].dropna()
    ax.scatter(i+np.linspace(-.15,.15,len(y)),y,color=group_ink[g],s=30)
    if len(y):
        ax.hlines(y.median(),i-.23,i+.23,color="#333")
ax.axhline(0,color="#777",lw=.7)
ax.set(xticks=range(3),xticklabels=group_order,ylim=(-1,1),ylabel="Within-donor Spearman ρ",
       title="Composition DPT vs. measured core + peri-islet immune density")
fig.tight_layout()
plt.show()
display(overview)